# Impulse — Getting Started

A minimal one-event, one-histogram report against the Seat Leon demo data. Run the cells top to bottom on a serverless Databricks cluster — fill in the **Catalog**, **Schema**, and **Table Prefix** widgets that appear after running cell 1.

Full walkthrough at the [Getting Started page](https://databrickslabs.github.io/impulse/docs/getting_started).

In [ ]:
dbutils.widgets.text("catalog", "", "Catalog")
dbutils.widgets.text("schema", "", "Schema")
dbutils.widgets.text("table_prefix", "", "Table Prefix")

In [ ]:
import sys, os
import pandas as pd

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
TABLE_PREFIX = dbutils.widgets.get("table_prefix")

if not CATALOG or not SCHEMA or not TABLE_PREFIX:
    raise ValueError("Please set Catalog, Schema, and Table Prefix widgets above before running.")

nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
DEMOS_DIR = "/Workspace" + "/".join(nb_path.split("/")[:-1])
REPO_ROOT = "/Workspace" + "/".join(nb_path.split("/")[:-2])
sys.path.insert(0, os.path.join(REPO_ROOT, "src"))

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

csv_dir = os.path.join(DEMOS_DIR, "data", "reporting")
for t in ["container_metrics", "container_tags", "channel_metrics", "channel_tags", "channels"]:
    (spark.createDataFrame(pd.read_csv(f"{csv_dir}/{t}.csv"))
          .write.mode("overwrite")
          .saveAsTable(f"{CATALOG}.{SCHEMA}.{TABLE_PREFIX}_{t}"))
print(f"Loaded 5 silver-layer tables under {CATALOG}.{SCHEMA}.{TABLE_PREFIX}_*")

In [ ]:
from databricks.sdk import WorkspaceClient

from impulse_reporting.aggregations.histogram import HistogramDuration
from impulse_reporting.core.page import Page
from impulse_reporting.core.report import Report
from impulse_reporting.events.basic_event import BasicEvent

pfx = f"{CATALOG}.{SCHEMA}.{TABLE_PREFIX}"

config = {
    "source": {
        "container_metrics_table": f"{pfx}_container_metrics",
        "container_tags_table":    f"{pfx}_container_tags",
        "channel_metrics_table":   f"{pfx}_channel_metrics",
        "channel_tags_table":      f"{pfx}_channel_tags",
        "channels_uri":            f"{pfx}_channels",
    },
    "unity_sink": {
        "catalog": CATALOG,
        "schema":  SCHEMA,
        "table_prefix": TABLE_PREFIX,
    },
    "query_engine": {"solver": "DeltaSolver", "data_type": "RAW"},
    "measurement_dimensions": ["container_id", "vehicle_key", "start_ts", "stop_ts"],
}

report = Report(
    name="getting_started",
    spark=spark,
    workspace_client=WorkspaceClient(),
    config=config,
)
db = report.get_db()

eng_rpm = db.query.channel(channel_name="Engine RPM", brand="Seat", model="Leon")

high_rpm = BasicEvent(
    name="high_rpm",
    expr=eng_rpm > 2000,
    desc="Engine RPM above 2000",
)
report.add_event(high_rpm)

page = Page(page_number=1)
page.add_aggregation(
    HistogramDuration(
        name="rpm_distribution",
        base_expr=eng_rpm,
        bins=[float(x) for x in range(0, 5001, 500)],
        event=high_rpm,
        channel_name="Engine RPM",
        bins_unit="rpm",
        values_unit="s",
    )
)
report.add_page(page)

report.determine_report()
report.persist_results()


In [0]:
import pyspark.sql.functions as F

display(
    spark.read.table(f"{pfx}_histogram_fact")
    .groupBy("bin_name", "lower_bound")
    .agg(F.sum("hist_value").alias("duration_us"))
    .orderBy("lower_bound")
)


bin_name,lower_bound,duration_us
0.0-500.0,0.0,0.0
500.0-1000.0,500.0,0.0
1000.0-1500.0,1000.0,0.0
1500.0-2000.0,1500.0,0.0
2000.0-2500.0,2000.0,2.985225005E9
2500.0-3000.0,2500.0,3.05175E8
3000.0-3500.0,3000.0,7.6034E7
3500.0-4000.0,3500.0,0.0
4000.0-4500.0,4000.0,0.0
4500.0-5000.0,4500.0,0.0


Databricks visualization. Run in Databricks to view.